# Task-decoupled federated continual learning (Tasks 1 & 4) on Kaggle

**What this runs** (`PLAN = "novel+mini"`, both T4 GPUs, about 3 h):
* **GPU 0, `novel`** – the proposed method:
  1. *Task-decoupled FL*: FedAvg with duration-only / disposal-only / all heads kept local per court (shared FedAvg comes from the sweep); best sharing choice picked on validation loss.
  2. *Federated continual onboarding*: courts join one at a time with that sharing choice, comparing naive, EWC, replay, EWC+replay; every joined court is re-evaluated after each new court, so forgetting is measured. Best combination (by **validation** forgetting) is re-run with Madhya Pradesh joining first.
* **GPU 1, `mini`** – seeds 1 and 2 of FedAvg, FedAvg+fine-tuning, local and centralized, for the ± in the main table.

**Before running**
1. Dataset (attached via **Add Input**) must contain `judgments_labelled.csv`, the **new** `fl_judicial.py`, `run_queue.py`, `summarize_results.py`, and the `runs/` folder from the sweep (or `judicial_fl_resume_dataset.zip`).
2. Settings: **GPU T4 x2**, **Internet On**.
3. **Save Version → Save & Run All (Commit)**. Finished runs are skipped; interrupted FL runs resume.

Outputs: `runs/all_results.csv`, `comparison.csv`, `forgetting_overview.csv`, `forgetting_curves.png`, `per_court_*.png`.

In [ ]:
!pip install -q peft
!pip uninstall -y -q torchao
import os, glob, shutil, subprocess, sys, time, torch
MARK = {"fl_judicial.py": "fed_continual", "run_queue.py": '"novel"', "summarize_results.py": "sel_val_loss"}
def find(name):
    """newest copy: if several versions are attached, take the one with the current features"""
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    good = [h for h in hits if MARK.get(name, "") in open(h).read()]
    return (good or hits or [None])[0]
# the CSV may have been uploaded under a slightly different name (e.g. judgments_labelled_1.csv)
csvs = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
cands = [p for p in csvs if "judgments_labelled" in os.path.basename(p)]
DATA = cands[0] if cands else None
assert DATA, f"judgments_labelled*.csv not found under /kaggle/input. CSVs found: {csvs}"
for f in ["fl_judicial.py", "run_queue.py", "summarize_results.py"]:
    p = find(f); assert p, f"{f} not found under /kaggle/input: upload it to your dataset"
    shutil.copy(p, f"/kaggle/working/{f}")
os.chdir("/kaggle/working")
N_GPU = torch.cuda.device_count()
print("data:", DATA, "| GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
assert "fed_continual" in open("fl_judicial.py").read(), "old fl_judicial.py in the dataset: upload the new one"
assert '"novel"' in open("run_queue.py").read(), "old run_queue.py in the dataset (no 'novel' plan): upload the new one"
assert N_GPU >= 1, "no GPU: set Accelerator = GPU T4 x2"
import peft, urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=10); print("internet: ok")
except Exception as e:
    raise RuntimeError(f"no internet ({e}): turn Internet On (phone-verified account needed)")

In [ ]:
# Resume: restore runs/ (results + checkpoints) from any attached input
import zipfile
os.makedirs("runs", exist_ok=True)
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):          # runs uploaded as a zip
    with zipfile.ZipFile(z) as f:
        if any(n.endswith("all_results.csv") for n in f.namelist()):
            f.extractall("/kaggle/working/_restore"); print("unzipped", z)
for p in glob.glob("/kaggle/input/**/all_results.csv", recursive=True) + \
         glob.glob("/kaggle/working/_restore/**/all_results.csv", recursive=True):
    shutil.copytree(os.path.dirname(p), "runs", dirs_exist_ok=True)
    print("restored", os.path.dirname(p))
done = [d for d in os.listdir("runs") if os.path.exists(f"runs/{d}/per_court_test.csv")]
partial = [d for d in os.listdir("runs") if os.path.exists(f"runs/{d}/ckpt.pt") and d not in done]
print(f"finished runs: {len(done)} | interrupted runs that will resume: {partial}")
assert os.path.exists("runs/all_results.csv"), "no previous results found: check the attached dataset"

In [ ]:
PLAN = "novel+mini"      # "novel+mini" (GPU0 novel, GPU1 extra seeds) | "novel" | "mini" | "sweep" | "confirm"
BUDGET_HOURS = 3.2 if PLAN in ("novel+mini", "novel", "mini") else 10.8   # stop launching new runs after this
ENCODERS = ["roberta-base", "law-ai/InLegalBERT"]

def launch(gpu, plan, extra, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = [sys.executable, "-u", "run_queue.py", "--data", DATA, "--plan", plan, "--budget_hours", str(BUDGET_HOURS), *extra]
    return subprocess.Popen(cmd, env=env, stdout=open(log, "a"), stderr=subprocess.STDOUT)

t0 = time.time(); procs = []
if PLAN == "novel+mini" and N_GPU >= 2:
    procs = [launch(0, "novel", [], "queue_gpu0.log"), launch(1, "mini", ["--seeds", "1,2"], "queue_gpu1.log")]
elif PLAN == "novel+mini":           # one GPU: novel first (the main contribution), then seeds if time is left
    procs = [launch(0, "novel", [], "queue_gpu0.log")]
elif PLAN == "sweep" and N_GPU >= 2:
    procs = [launch(i, "sweep", ["--encoder", e], f"queue_gpu{i}.log") for i, e in enumerate(ENCODERS[:2])]
elif PLAN == "mini" and N_GPU >= 2:
    procs = [launch(i, "mini", ["--seeds", str(i + 1)], f"queue_gpu{i}.log") for i in range(2)]
elif PLAN == "confirm" and N_GPU >= 2:
    procs = [launch(i, "confirm", ["--shard", str(i), "--n_shards", "2"], f"queue_gpu{i}.log") for i in range(2)]
else:
    extra = {"sweep": ["--encoder", ENCODERS[1]], "mini": ["--seeds", "1,2"], "confirm": []}.get(PLAN, [])
    procs = [launch(0, PLAN, extra, "queue_gpu0.log")]
print("launched", len(procs), "queue(s):", PLAN)

In [ ]:
# Monitor until the queues finish (prints progress every 10 minutes)
import re
while any(p.poll() is None for p in procs):
    time.sleep(600)
    print(f"\n===== {(time.time()-t0)/3600:.2f} h =====")
    for log in sorted(glob.glob("queue_gpu*.log")):
        lines = [l for l in open(log).read().splitlines() if re.search(r"start|mean:|chosen|choice|least VALID|stage|early stop|best round|Error|skipped", l)]
        print(log, "\n  " + "\n  ".join(lines[-6:]))
print("all queues finished")
for log in sorted(glob.glob("queue_gpu*.log")):
    print(open(log).read()[-3000:])
# fail loudly if any run crashed (otherwise the summary silently shows only old results)
errs = []
for log in glob.glob("logs/*.log") + glob.glob("queue_gpu*.log"):
    txt = open(log).read()
    if "Traceback" in txt or "Error" in txt:
        errs.append(log); print(f"\n!!!!! errors in {log}:\n" + txt[-2500:])
assert not errs, f"runs failed, see logs above: {errs}"

In [ ]:
!python summarize_results.py runs/all_results.csv

In [ ]:
# Proposed method: forgetting during federated continual onboarding (test set; configs were chosen on validation)
import pandas as pd, numpy as np, matplotlib.pyplot as plt, json
rows, curves = [], {}
for d in sorted(glob.glob("runs/fed_continual_*")):
    fs, fc = os.path.join(d, "forgetting_summary.csv"), os.path.join(d, "forgetting.csv")
    if not os.path.exists(fs): continue
    cfg = json.load(open(os.path.join(d, "config.json"))); s = pd.read_csv(fs); h = pd.read_csv(fc)
    last = h[h.stage == h.stage.max()]
    rows.append({"run": os.path.basename(d), "ewc_lambda": cfg["ewc_lambda"], "replay": cfg["replay_size"],
                 "local_heads": cfg.get("local_heads", "none"), "order": cfg.get("onboard_order") or "largest first",
                 "final_MAE_log": last.test_MAE_log.mean(), "final_disp_F1": last.test_disp_F1.mean(),
                 "worst_MAE_log": last.test_MAE_log.max(), "worst_disp_F1": last.test_disp_F1.min(),
                 "forgetting_MAE_log": s.MAE_log_forgetting.mean(), "forgetting_disp_F1": s.disp_F1_forgetting.mean(),
                 "max_forgetting_disp_F1": s.disp_F1_forgetting.max()})
    curves[os.path.basename(d)] = h
ov = pd.DataFrame(rows)
if len(ov):
    ov.to_csv("forgetting_overview.csv", index=False); display(ov.round(3))
    fig, axes = plt.subplots(1, len(curves), figsize=(4.2 * len(curves), 3.4), sharey=True)
    for ax, (name, h) in zip(np.atleast_1d(axes), curves.items()):
        for c, g in h.groupby("eval_court"):
            ax.plot(g.stage, g.test_disp_F1, marker="o", label=c)
        ax.set_title(name.replace("fed_continual_", "")[:38], fontsize=8); ax.set_xlabel("courts onboarded")
    np.atleast_1d(axes)[0].set_ylabel("test disposal F1"); np.atleast_1d(axes)[-1].legend(fontsize=7)
    plt.tight_layout(); plt.savefig("forgetting_curves.png", dpi=130); plt.show()
else:
    print("no fed_continual runs finished yet")

# Task-decoupled FL: sharing choices side by side (seed 0)
r = pd.read_csv("runs/all_results.csv")
cfgs = {d: json.load(open(f"runs/{d}/config.json")) for d in r.run.unique() if os.path.exists(f"runs/{d}/config.json")}
r["local_heads"] = r.run.map(lambda x: cfgs.get(x, {}).get("local_heads", "none") or "none")
dec = r[(r.method == "fedavg") & (r.seed == 0)]
g = dec.groupby(["model", "lr", "trunc", "local_heads"]).agg(val_loss=("sel_val_loss", "mean"), MAE_log=("MAE_log", "mean"),
        disp_F1=("disp_F1", "mean"), worst_MAE_log=("MAE_log", "max"), worst_disp_F1=("disp_F1", "min"))
display(g.round(3))

In [ ]:
# Per-court view of seed-0 runs at the chosen setting (best FedAvg on validation)
import pandas as pd, matplotlib.pyplot as plt
r = pd.read_csv("runs/all_results.csv"); r = r[(r.seed == 0) & (r.max_len == 256)]
fa = r[r.method == "fedavg"].groupby(["model", "lr", "trunc"]).sel_val_loss.mean()
for (enc, lr, tr) in fa.sort_values().index[:2]:
    s = r[(r.model == enc) & (r.lr == lr) & (r.trunc == tr)]
    s = s.assign(label=s.method + " | " + s.run.str.extract(r"(mu[\d.]+|beta-?[\d.]+|slr[\d.e-]+|m0\.5)", expand=False).fillna(""))
    fig, ax = plt.subplots(1, 2, figsize=(15, 4))
    for a, m in zip(ax, ["MAE_log", "disp_F1"]):
        s.pivot_table(index="court", columns="label", values=m).plot.bar(ax=a, legend=(m == "disp_F1"))
        a.set_title(f"{m} per court — {enc}, lr={lr}, {tr}"); a.set_xlabel("")
    ax[1].legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout(); plt.savefig(f"per_court_{enc.split('/')[-1]}.png", dpi=130); plt.show()